# Lekcia 11 - Protokol medzi agentmi (A2A)


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Čo je protokol A2A?

The **Agent-to-Agent (A2A) protokol** je otvorený štandard, ktorý umožňuje AI agentom komunikovať,
navzájom sa objavovať, a spolupracovať — aj keď sú postavené na rôznych frameworkoch alebo hosťované
rôznymi službami.

Key concepts:

- **Objavovanie** – Agenti zverejňujú *Kartu agenta*, ktorá popisuje ich schopnosti, čo
  ostatným agentom (alebo orchestrátorom) uľahčuje nájsť správneho špecialistu pre úlohu.
- **Prenos správ** – Agenti si vymieňajú štruktúrované správy prostredníctvom spoločného protokolu, takže
  požiadavka od jedného agenta môže byť pochopená a splnená iným bez ohľadu na jeho vnútornú
  implementáciu.
- **Životný cyklus úlohy** – A2A definuje stavy ako *odoslané*, *spracováva sa*, *dokončené*, a
  *zlyhalo*, čo orchestrátorovi poskytuje úplný prehľad o tom, ako postupuje delegovaná úloha.

V tejto lekcii simulujeme spoluprácu v štýle A2A prepojením troch špecializovaných cestovných agentov
do pracovného toku, kde každý agent prispeje svojou odbornosťou a odovzdá výsledky ďalšiemu.


## Vytváranie špecializovaných cestovných agentov


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Spolupráca viacerých agentov cez pracovný tok

Prepojíme tri agenty do sekvenčného pracovného toku, ktorý zodpovedá A2A odovzdávaniu správ:

1. **CurrencyExchangeAgent** dostane požiadavku používateľa a vygeneruje odporúčania týkajúce sa meny.
2. **ActivityPlannerAgent** dostane obohatený kontext a pridá odporúčania aktivít.
3. **TravelManagerAgent** zhŕňa oba vstupy do konečného cestovného zhrnutia.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pochopenie A2A v produkčnom prostredí

V produkčnom prostredí protokol A2A odomyká silné medzislužbové scenáre:

| Schopnosť | Popis |
|---|---|
| **Medzirámcová interoperabilita** | Agent vytvorený v jednom frameworku môže delegovať úlohy agentovi vytvorenému v akomkoľvek inom rámci kompatibilnom s A2A, čo umožňuje skutočnú interoperabilitu medzi organizáciami. |
| **Hranice služieb** | Agenti môžu bežať v samostatných mikroservisoch, cloudových regiónoch alebo dokonca v rôznych organizáciách a pritom bezproblémovo spolupracovať. |
| **Dynamické objavovanie** | Orchestrátor môže počas behu dotazovať register Agent Card, aby našiel najvhodnejšieho špecialistu pre danú podúlohu. |
| **Streaming a push notifikácie** | A2A podporuje Server-Sent Events (SSE) pre aktualizácie priebehu v reálnom čase a push notifikácie pre dlhodobé úlohy. |

Pracovný tok, ktorý sme vytvorili vyššie, je zjednodušenou, v-procesnou verziou tohto vzoru. V skutočnom nasadení by každý agent vystavil HTTP koncový bod, publikoval Agent Card a komunikoval cez protokol A2A JSON-RPC.


## Zhrnutie

V tejto lekcii ste sa naučili:

1. **Čo je protokol A2A** — otvorený štandard pre objavovanie medzi agentmi, zasielanie správ,
   a správa úloh.
2. **Ako vytvoriť špecializovaných agentov** — agenta Currency Exchange, agenta Activity Planner,
   a orchestrátora Travel Manager.
3. **Ako zapojiť agentov do pracovného postupu** — pomocou `WorkflowBuilder` na modelovanie sekvenčného
   prenosu správ medzi agentmi.
4. **Ako A2A funguje v produkcii** — umožňuje spoluprácu naprieč rámcami a službami
   s dynamickým objavovaním a streamovanými aktualizáciami.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Vylúčenie zodpovednosti:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kriticky dôležité informácie odporúčame profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia ani nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
